# Weather Forecast Modeling V1.1 — T2H

This notebook runs the isolated T2H contract on Colab. It pins the project branch, mounts Drive, resumes source chunks, builds the 73-feature dataset, freezes on VALIDATION, and evaluates TEST once. Drive paths include the run ID so T1H artifacts remain separate.

Predictors: Open-Meteo Historical Forecast API, `models=ecmwf_ifs`. Target: ERA5 `temperature_2m` at exact same-location `feature_time + 2h`. ERA5 is a reanalysis reference, not station truth. Historical Forecast vintage does not expose row-level run or lead time. The downloader enforces rolling free-tier minute/hour unit limits and a daily budget; quota waits are recorded in the manifest, and completed chunks resume by checksum. If the daily budget defers chunks, rerun this notebook after quota reset.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
assert MY_DRIVE.is_dir(), 'Google Drive mount is not available'

In [ ]:
import subprocess

REPO_URL = 'https://github.com/HaoLe2710/weather-streaming-bigdata.git'
BRANCH = 'model/weather-forecast-xgboost-t2h-v1'
REPO = Path('/content/weather-streaming-bigdata')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import os
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-ml.txt')], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print('Repository:', REPO)

In [ ]:
import shutil

RUN_ID = '20261003T070520Z-xgb-t2h-v1-1'
DRIVE_PROJECT = MY_DRIVE / 'weather-streaming-bigdata'
DATA_ROOT = DRIVE_PROJECT / 'datasets' / 'historical_forecast_t2h_v1_1' / RUN_ID
DATASET_ROOT = DRIVE_PROJECT / 'datasets' / 'weather_forecast_fe_t2h_v1_1' / RUN_ID
ARTIFACT_ROOT = DRIVE_PROJECT / 'artifacts' / 'weather_forecast_xgboost_t2h_v1_1' / RUN_ID
MODEL_PATH = ARTIFACT_ROOT / 'weather_forecast_xgboost_t2h_v1_1.json'
for directory in (DATA_ROOT, DATASET_ROOT, ARTIFACT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
SOURCE_EVIDENCE = REPO / 'results' / 'modeling-t2h' / RUN_ID
for name in ('provider_audit.json', 'source_contract.json'):
    destination = ARTIFACT_ROOT / name
    source = SOURCE_EVIDENCE / name
    if destination.exists() and destination.read_bytes() != source.read_bytes():
        raise RuntimeError(f'Existing Drive evidence differs: {destination}')
    if not destination.exists():
        shutil.copy2(source, destination)
print('Run:', RUN_ID)
print('Historical data:', DATA_ROOT)
print('ML dataset:', DATASET_ROOT)
print('Model/evidence:', ARTIFACT_ROOT)

In [ ]:
from historical.forecast_t2h_v1_1 import download_range, validate_normalized_sources

download_manifest = download_range(
    data_root=DATA_ROOT,
    artifact_root=ARTIFACT_ROOT,
    start_date=__import__('datetime').date(2020, 1, 1),
    end_date=__import__('datetime').date(2025, 12, 31),
    batch_size=10,
    max_retries=3,
    request_delay_seconds=30,
    max_daily_api_call_units=10000,
)
print({key: download_manifest.get(key) for key in ('status', 'completed_chunk_count', 'failed_chunk_count', 'deferred_chunk_count', 'estimated_api_call_units_used_today')})
if download_manifest.get('deferred_chunk_count', 0):
    raise RuntimeError('Open-Meteo daily quota deferred chunks. Rerun this cell after quota reset; completed chunks will be resumed.')
raw_validation = validate_normalized_sources(
    data_root=DATA_ROOT,
    artifact_root=ARTIFACT_ROOT,
    start_date=__import__('datetime').date(2020, 1, 1),
    end_date=__import__('datetime').date(2025, 12, 31),
)
assert raw_validation['status'] in ('PASS', 'PASS_WITH_GAPS'), raw_validation['errors']

from historical.forecast_t2h_v1_1 import probe_live_forecast_alignment
live_probe = probe_live_forecast_alignment(artifact_root=ARTIFACT_ROOT)
assert live_probe['locations'] == 63 and live_probe['comparison']['coordinate_mismatches'] == 0
from ml.forecast_t2h_v1_1 import write_source_alignment
source_alignment = write_source_alignment(ARTIFACT_ROOT, ARTIFACT_ROOT / 'live_forecast_probe.json')
print('Live Best Match vs explicit IFS:', live_probe['comparison']['exact_match'])


In [ ]:
from ml.forecast_t2h_v1_1 import materialize_feature_dataset

dataset_manifest = materialize_feature_dataset(
    data_root=DATA_ROOT,
    dataset_root=DATASET_ROOT,
    artifact_root=ARTIFACT_ROOT,
    start_date=__import__('datetime').date(2020, 1, 1),
    end_date=__import__('datetime').date(2025, 12, 31),
)
print({key: dataset_manifest[key] for key in ('total_rows', 'rows_by_split', 'feature_count', 'feature_list_sha256')})

from ml.forecast_t2h_v1_1 import write_distribution_comparison
source_alignment = write_source_alignment(ARTIFACT_ROOT, ARTIFACT_ROOT / 'live_forecast_probe.json')
distributions = write_distribution_comparison(DATASET_ROOT, ARTIFACT_ROOT / 'live_forecast_probe.json', ARTIFACT_ROOT)
print('TRAIN vs current live distributions:', distributions['status'], distributions['training_row_count'])


In [ ]:
from ml.train_forecast_t2h_v1_1 import train_and_evaluate, write_checksums

if (ARTIFACT_ROOT / 'test_metrics.json').exists():
    test_metrics = __import__('json').loads((ARTIFACT_ROOT / 'test_metrics.json').read_text())
    print('TEST was already evaluated once; reusing the persisted report.')
else:
    test_metrics = train_and_evaluate(
        dataset_root=DATASET_ROOT,
        artifact_root=ARTIFACT_ROOT,
        model_path=MODEL_PATH,
    )
checksums = write_checksums(ARTIFACT_ROOT, MODEL_PATH)
print(test_metrics)
print('Checksum entries:', len(checksums['files']))
print('Model bytes:', MODEL_PATH.stat().st_size)